# 04 — Backtesting & risk metrics

**Purpose:** Align with `metrics.py`: **Sharpe**, annualized return/vol from $(\mu, \Sigma, w)$, and **max drawdown** + equity curves from `run_backtest` (hold-out tail of the window vs equal-weight benchmark).


In [ ]:
# Allow imports from backend/ (start Jupyter from `notebooks/` or project root)
from pathlib import Path
import sys
for base in [Path.cwd(), Path.cwd() / "notebooks"]:
    for b in [base, base.parent]:
        be = b / "backend"
        if (be / "optimize.py").exists():
            sys.path.insert(0, str(be))
            break
    else:
        continue
    break
else:
    raise RuntimeError("Could not find backend/ — `cd` to smart-portfolio-system or notebooks/")


In [ ]:
from optimize import run_optimization
from metrics import compute_metrics, run_backtest

TICKERS = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA"]
START, END = "2022-01-01", "2024-12-31"

weights_dict, mu, cov = run_optimization(TICKERS, START, END, lambda_risk=5)
m = compute_metrics(weights_dict, mu, cov)
m


In [ ]:
bt = run_backtest(TICKERS, weights_dict, START, END, test_split=0.2)
bt["max_drawdown"], len(bt["equity_curve"])


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

eq = pd.Series({p["date"]: p["value"] for p in bt["equity_curve"]})
bm = pd.Series({p["date"]: p["value"] for p in bt["benchmark_curve"]})
ax = eq.plot(label="optimized (simple cumulative)", figsize=(10, 4))
bm.plot(ax=ax, label="equal-weight benchmark")
ax.legend()
ax.set_title("Normalized equity (hold-out slice — see metrics.run_backtest)")
plt.tight_layout()
plt.show()


### Caveat
`run_backtest` uses the **last** fraction of trading days (`test_split`); it is a **simple** historical simulation, not a full walk-forward study.
